## Week 4 Notebook 3: RAG eval with LLM-as-a-judge
In this notebook, we'll evaluate: 
* LLM-as-a-judge

-------

For each generated question, we
run RAG and save the answer produced by the LLM. Later, we'll compare
this answer with the original FAQ answer.

This is the A->Q->A' setup:

- A = original answer in the FAQ
- Q = generated question from this answer
- A' = answer produced by our RAG system

If A' is close to A, the RAG system is doing a good job.

This is still offline evaluation. We can compare A and A' because our
questions came from FAQ records. For each question, we know which
original answer it came from.


In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

Load the FAQ documents and the search index:

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

Create a lookup table for the original FAQ docs:

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

#### Running RAG

In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [5]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

Run RAG for 1 question:

In [6]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes — you can still join the course.\n\nAccording to the course info, you can start learning and submitting homework while the form is open, and you don’t need special confirmation. If you want a certificate, you just need to submit your project while submissions are still being accepted.'

check cost of this execution:

In [7]:
assistant.total_cost()

0.0006405

Get the original answers from the doc ID:

In [8]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

Now save both answers in one record: 

In [9]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'I found this course late — can I still start it now, or is it already closed?',
 'answer_llm': 'Yes — you can still join the course.\n\nAccording to the course info, you can start learning and submitting homework while the form is open, and you don’t need special confirmation. If you want a certificate, you just need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

Processing all questions

* Create a function that processes one grouth truth record

In [10]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [11]:
# Test on one record
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'I found this course late — can I still start it now, or is it already closed?',
 'answer_llm': 'Yes, you can still join and start now — but if you want a certificate, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [12]:
# Before running the full batch, reset the usage we collected while testing:
assistant.reset_usage()

In [13]:
# We now run RAG for all ground truth questions with parallel processing
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(
        pool, ground_truth, generate_rag_answer
    )  # generate_rag_answer returns one answer record for each question

  0%|          | 0/515 [00:00<?, ?it/s]

In [14]:
# Collect all answers
answers = []

for answer_record in results:
    answers.append(answer_record)

In [ ]:
# calculate total cost
assistant.total_cost()

0.5257522500000007

In [ ]:
# save the answers to a csv file
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)

## LLM-as-a-judge

In [7]:
# Load data
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [8]:
from pydantic import BaseModel, Field
from typing import Literal


class AnswerEvaluation(BaseModel):
    reasoning: str = Field(description="Reasoning about the quality of the answer.")
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [9]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

Then define the prompt template. This is the data we pass to the judge for each answer.

In [10]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

LLM Answer:
{answer_llm}
""".strip()

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import (
    calc_price,
    calc_total_price,
    llm_structured_retry,
    map_progress,
)

load_dotenv()
openai_client = OpenAI()

In [13]:
# apply llm-as-a-judge on one record
rec = answers[0]
rec

{'question': 'I found this course late — can I still start it now, or is it already closed?',
 'answer_llm': 'Yes — you can still join and start learning.\n\nIf you want a certificate, make sure you submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [20]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"],
)

print(prompt)

Question:
I found this course late — can I still start it now, or is it already closed?

Original Answer (ground truth):
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

LLM Answer:
Yes — you can still join and start learning.

If you want a certificate, make sure you submit your project while submissions are still being accepted.


In [18]:
eval_result, usage = llm_structured_retry(
    openai_client, aqa_judge_instructions, prompt, AnswerEvaluation
)
eval_result

AnswerEvaluation(reasoning='The AI answer preserves the core meaning of the ground truth: late enrollment is still possible, and certificate eligibility requires submitting the project before submissions close. It adds a bit of paraphrasing but no contradiction or missing key point.', score='good')

In [21]:
calc_price(usage)

{'input_cost': 0.000225,
 'output_cost': 0.0002745,
 'total_cost': 0.0004994999999999999}

putting the same logic in a function

In [ ]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question, answer_orig=answer_orig, answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [ ]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"],
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key message: the course can still be started, and certificate eligibility depends on submitting the project before submissions close. This is semantically equivalent to the ground truth.', score='good')

Run the LLM-as-a-judge on all answers:

In [ ]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"],
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

Using the same parallel processing helper

In [28]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/515 [00:00<?, ?it/s]

Split the results into eval vs. usage, and then create a dataframe of the evals

In [30]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)


df_eval = pd.DataFrame(evaluations)

In [ ]:
# check the results - % of "good" vs. "bad" answers
df_eval.score.value_counts(normalize=True)

score
good    0.957282
bad     0.042718
Name: proportion, dtype: float64

In [33]:
# look at the "bad" cases to understand what went wrong:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
1,Is it okay to join the course after it has alr...,74eb249bbf,bad,"The ground truth says yes, late joining is all..."
2,"If I enroll now, will I still be able to get a...",74eb249bbf,bad,"The ground truth says yes, a certificate is st..."
29,Do I need to take part in peer review to quali...,69d122f12e,bad,The ground truth says you do not get a certifi...
34,Will missing one homework prevent me from fini...,9f689c185f,bad,The ground truth says missing one homework wil...
42,Is there a planned date for the next run of th...,bd31146b0e,bad,The ground truth gives a specific planned date...


In [31]:
# calculate the total cost:
calc_total_price(usages)

0.3588787499999997

In [ ]:
# save the results to csv
df_eval.to_csv("data/rag_evals.csv", index=False)

## Evaluating the judge

* The judge can be wrong. It may rate an answer as good even though search
failed to retrieve the right document. In that case the judge is too
lenient. Make the instructions stricter and re-run the evaluation.

* To evaluate the judge, you need to look at the results yourself. Sample
some good and bad cases, read the judge reasoning, and check whether you
agree with the verdict. You cannot use another judge to evaluate the
judge. This is manual work, but it is necessary.

* A practical approach is to build a simple application with `Streamlit`.
Show each question, the original answer, the generated answer, and the
judge verdict side by side. Then mark each verdict as correct or
incorrect and use that feedback to adjust the judge instructions. This
is a lot of trial and error, but it makes the evaluation framework more
reliable.